In [1]:
import os
import sys
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
#import nipes_data_fcts as nidf
import me_data_fcts as medf
#from statannotations.Annotator import Annotator
import scipy.sparse
import sympy
import sklearn.datasets
import sklearn.feature_extraction.text
import umap
import umap.plot
import matplotlib.pyplot as plt
from numpy.random import rand
from numpy import genfromtxt
%matplotlib inline

/usr/local/lib/python3.10/dist-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
/usr/local/lib/python3.10/dist-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.NumbaWarning)
/usr/local/lib/python3.10/dist-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.

In [2]:
repo_folder = "/are/logs/are/meim_env_pressure_alife/10k_runs/"
experiments = {
    "directory" :  ['meimflat', 'meimhill', 'meimholes', 'meimobs', 'meimrough', 'meimholly' ],
    "name" :       ['flat', 'hill', 'holes', 'obstacles', 'rough', 'hollyrood'],
    "print_name" : ['Flat', 'Hill', 'Holes', 'Obstacles', 'Rough', 'Hollyrood'],
}

In [3]:
def load_sparse_morph(name, folders):
  _robots_loaded = 0
  _morph_sparse = []
  _morph_fitness = np.empty((0))
  _morph_labels = []
  for folder in folders:
      print(folder)
      _morph_dense = genfromtxt(folder + '/morph_descriptor.csv', delimiter=',')
      _morph_fit = genfromtxt(folder + '/fitness.csv', delimiter=',')[:,3]
      assert _morph_dense.shape[0] == _morph_fit.shape[0]
      print(_morph_dense.shape)
      _morph_fit = _morph_fit / np.max(_morph_fit)
      _morph_fitness = np.append(_morph_fitness, _morph_fit, axis=0)
      _robots_loaded += _morph_dense.shape[0]
      if _morph_sparse == []:
          _morph_sparse = scipy.sparse.lil_matrix(_morph_dense[:, 2:])
      else:
          temp_sparse = scipy.sparse.lil_matrix(_morph_dense[:, 2:])
          print("temp_sparse", temp_sparse.shape)
          _morph_sparse = scipy.sparse.vstack([_morph_sparse, temp_sparse])
      print("sparse", _morph_sparse.shape)
  _morph_labels = np.array([name] * _robots_loaded)
  return _morph_sparse, _morph_fitness, _morph_labels, _robots_loaded

In [4]:
def load_morphology(name, folders):
    exp_descs = []
    experiments_loaded = 0
    for folder in folders:
        #parent_ids = medf.load_parent_pool(folder + "/parent_pool.csv")
        descriptors = medf.load_feature_descriptor(folder + "/morph_features.csv")
        #print(descriptors[-1])
        descs = [[int(d[0])] + d[1:5] + [d[5]*16,
                                         d[6]*16,
                                         d[7]*16,
                                         d[8]*16]
                                           + d[9:]  + [folder] 
                 for d in descriptors]
        #print(descs[-1])
        #parent_descs += medf.filter_to_parent_pool(descs,parent_ids)
        exp_descs += descs        
        print(folder, " -- Robots: ", len(descs))
        experiments_loaded += 1
    data = pd.DataFrame(data=exp_descs,columns=[
                                                            "robot index",
                                                            "width",
                                                            "depth",
                                                            "height",
                                                            "voxels",
                                                            "wheels",
                                                            "sensors",
                                                            "limbs",
                                                            "casters",
                                                            "norm",
                                                            "skeleton norm",
                                                            "components norm",
                                                            "replicate_index", 
                                                            "replicate"])
    data["environment"] = name
    print("Total experiments loaded: ", experiments_loaded, "-- Total robots: ", len(data))
    return data

In [5]:
# loading descriptor data
robot_desc_data = []
for idx, exps in enumerate(experiments['directory']):
    print(exps)
    folders = []
    data = []
    for folder in os.listdir(repo_folder + exps):
        for fold in os.listdir(repo_folder + exps + '/' + folder):
            folders.append(repo_folder + exps + '/' + folder + '/' + fold)
    data = load_morphology(experiments['name'][idx], folders)
    robot_desc_data.append(data)

morph_features = pd.concat(robot_desc_data).reset_index(drop=True)

meimflat
/are/logs/are/meim_env_pressure_alife/10k_runs/meimflat/55742_8/meim_11_3_19-16-10-816-2370462392  -- Robots:  10046
/are/logs/are/meim_env_pressure_alife/10k_runs/meimflat/55742_9/meim_12_3_1-34-14-4030-2971763921  -- Robots:  10004
/are/logs/are/meim_env_pressure_alife/10k_runs/meimflat/56265_9/meim_18_3_6-1-36-6253-4195160045  -- Robots:  10000
/are/logs/are/meim_env_pressure_alife/10k_runs/meimflat/55742_7/meim_11_3_17-59-59-9003-1516430376  -- Robots:  10024
/are/logs/are/meim_env_pressure_alife/10k_runs/meimflat/56265_1/meim_17_3_21-46-8-8472-1119555096  -- Robots:  10048
/are/logs/are/meim_env_pressure_alife/10k_runs/meimflat/56265_8/meim_17_3_21-46-7-7298-717215794  -- Robots:  10000
/are/logs/are/meim_env_pressure_alife/10k_runs/meimflat/55742_10/meim_12_3_1-54-59-9138-4171306948  -- Robots:  10000
/are/logs/are/meim_env_pressure_alife/10k_runs/meimflat/55742_4/meim_11_3_17-59-57-7377-2053156979  -- Robots:  10036
/are/logs/are/meim_env_pressure_alife/10k_runs/meimfla

/are/logs/are/meim_env_pressure_alife/10k_runs/meimobs/55752_4/meim_13_3_5-20-44-4275-3596159210  -- Robots:  10034
/are/logs/are/meim_env_pressure_alife/10k_runs/meimobs/56277_2/meim_19_3_18-29-30-524-1885665388  -- Robots:  10031
/are/logs/are/meim_env_pressure_alife/10k_runs/meimobs/56277_10/meim_20_3_1-10-6-6258-3751677949  -- Robots:  10009
/are/logs/are/meim_env_pressure_alife/10k_runs/meimobs/56277_8/meim_19_3_21-47-46-6923-1649492607  -- Robots:  10044
/are/logs/are/meim_env_pressure_alife/10k_runs/meimobs/55752_2/meim_12_3_23-11-36-6655-2590764412  -- Robots:  10007
/are/logs/are/meim_env_pressure_alife/10k_runs/meimobs/55752_7/meim_13_3_6-23-17-7247-2149868613  -- Robots:  10031
/are/logs/are/meim_env_pressure_alife/10k_runs/meimobs/55752_8/meim_13_3_7-38-7-7759-1086957889  -- Robots:  10001
/are/logs/are/meim_env_pressure_alife/10k_runs/meimobs/56277_4/meim_19_3_20-1-53-3458-1133913684  -- Robots:  10032
Total experiments loaded:  19 -- Total robots:  190333
meimrough
/are/l

In [6]:
morph_labels = []
morph_sparse = []
morph_fitness = np.empty((0))
robots_loaded = 0
for idx, exps in enumerate(experiments['directory']):
    print(exps)
    folders = []
    data = []
    for folder in os.listdir(repo_folder + exps):
        for fold in os.listdir(repo_folder + exps + '/' + folder):
            folders.append(repo_folder + exps + '/' + folder + '/' + fold)
    _ms, _mf, _ml, _rl = load_sparse_morph(experiments['name'][idx], folders)
    robots_loaded += _rl
    if morph_sparse == []:
      morph_sparse = _ms
    else:
      morph_sparse = scipy.sparse.vstack([morph_sparse, _ms])
    print("morph_sparse", morph_sparse.shape)
    morph_fitness = np.append(morph_fitness, _mf, axis=0)
    morph_labels = np.append(morph_labels, _ml, axis=0)
print(morph_fitness.shape)
print(len(morph_labels))
print(morph_sparse.shape)
print(robots_loaded) 

meimflat
/are/logs/are/meim_env_pressure_alife/10k_runs/meimflat/55742_8/meim_11_3_19-16-10-816-2370462392
(10046, 1333)
sparse (10046, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimflat/55742_9/meim_12_3_1-34-14-4030-2971763921
(10004, 1333)
temp_sparse (10004, 1331)
sparse (20050, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimflat/56265_9/meim_18_3_6-1-36-6253-4195160045
(10000, 1333)
temp_sparse (10000, 1331)
sparse (30050, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimflat/55742_7/meim_11_3_17-59-59-9003-1516430376
(10024, 1333)
temp_sparse (10024, 1331)
sparse (40074, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimflat/56265_1/meim_17_3_21-46-8-8472-1119555096
(10048, 1333)
temp_sparse (10048, 1331)
sparse (50122, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimflat/56265_8/meim_17_3_21-46-7-7298-717215794
(10000, 1333)
temp_sparse (10000, 1331)
sparse (60122, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimflat/55742_10

(10022, 1333)
temp_sparse (10022, 1331)
sparse (130183, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimholes/55750_5/meim_12_3_3-42-56-6483-2933250927
(10010, 1333)
temp_sparse (10010, 1331)
sparse (140193, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimholes/55750_4/meim_12_3_2-40-20-317-4084338367
(10001, 1333)
temp_sparse (10001, 1331)
sparse (150194, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimholes/56275_10/meim_19_3_1-34-3-3720-466742052
(10003, 1333)
temp_sparse (10003, 1331)
sparse (160197, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimholes/55750_2/meim_12_3_2-0-19-9926-2921417019
(10006, 1333)
temp_sparse (10006, 1331)
sparse (170203, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimholes/55750_3/meim_12_3_2-11-21-1044-3216600959
(10030, 1333)
temp_sparse (10030, 1331)
sparse (180233, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimholes/55750_10/meim_12_3_12-0-0-298-1511744999
(10031, 1333)
temp_sparse (10031, 1331

(10009, 1333)
temp_sparse (10009, 1331)
sparse (70052, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimholly/56276_1/meim_19_3_1-45-4-4538-2613446622
(10001, 1333)
temp_sparse (10001, 1331)
sparse (80053, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimholly/56214_7/meim_16_3_6-42-25-5827-1007822800
(10000, 1333)
temp_sparse (10000, 1331)
sparse (90053, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimholly/56214_6/meim_15_3_23-21-22-2475-1802014424
(10000, 1333)
temp_sparse (10000, 1331)
sparse (100053, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimholly/56214_3/meim_15_3_23-21-24-4108-1558951991
(10047, 1333)
temp_sparse (10047, 1331)
sparse (110100, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimholly/56276_2/meim_19_3_2-50-9-9246-1746217039
(10024, 1333)
temp_sparse (10024, 1331)
sparse (120124, 1331)
/are/logs/are/meim_env_pressure_alife/10k_runs/meimholly/56276_10/meim_19_3_12-46-58-8695-1745531171
(10000, 1333)
temp_sparse (10000, 1

In [7]:
print(morph_fitness.shape)
print(morph_labels.shape)
print(morph_sparse.shape)
print(robots_loaded) 
np.min(morph_fitness)

(1163792,)
(1163792,)
(1163792, 1331)
1163792


0.0

In [ ]:
%%time
mapper = umap.UMAP(metric='manhattan', low_memory=True).fit(morph_sparse)

/usr/local/lib/python3.10/dist-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [ ]:
full_plot = umap.plot.points(mapper, labels=morph_labels, color_key_cmap='tab10', background='floralwhite')
plt.savefig("img_alife25/umap_full.png", bbox_inches='tight')
#full_plot_xlim = full_plot.get_xlim()
#full_plot_ylim = full_plot.get_ylim()
#full_plot.get_xlim()
#full_plot.get_ylim()

In [ ]:
ss = np.greater(morph_fitness, 0.6)
umap.plot.points(mapper, labels=morph_labels, subset_points=ss, color_key_cmap='tab10', background='floralwhite', alpha=0.9) #values=np.arange(morph_sparse.shape[0]), )
plt.savefig("img_alife25/umap_above_0.6_fitness.png", bbox_inches='tight')
#the_plot.set_xlim(full_plot_xlim)
#the_plot.set_ylim(full_plot_ylim)
#print(the_plot.get_xlim()[0])
#print(full_plot_xlim)

In [ ]:
for n in experiments['name']:
    ss = np.equal(morph_labels, n) * np.greater(morph_fitness, 0.5)
    the_plot = umap.plot.points(mapper, values=morph_fitness, subset_points=ss, color_key_cmap='tab10', background='floralwhite') #values=np.arange(morph_sparse.shape[0]), )
    the_plot.set_title(n)
#the_plot.set_xlim(-0.5,799.5)
#the_plot.set_ylim(799.5,-0.5)

In [ ]:
ss = np.greater(morph_fitness, 0.5)
#ss_fit = morph_fitness[ss > 0]
#the_plot = umap.plot.points(mapper, labels=morph_labels, subset_points=ss, theme='darkred',alpha=0.9) #values=np.arange(morph_sparse.shape[0]), )

p = umap.plot.interactive(mapper, subset_points=ss, labels=morph_labels, hover_data=morph_fitness, point_size=2)
umap.plot.show(p)


In [ ]:
ss = np.greater(morph_fitness, 0.6)
hover_data = morph_features.loc[:, ['robot index', 'width', 'depth', 'height', 'voxels', 'wheels', 'sensors', 'limbs', 'casters', 'norm', 'skeleton norm', 'components norm', 'replicate_index', 'replicate', 'environment']]
#interactive_plot = umap.plot.interactive(mapper, labels=morph_labels, hover_data=hover_data, point_size=2)
# Subsample the data to reduce the number of points for hover data
#subsample_indices = np.random.choice(len(hover_data), size=10000, replace=False)
#subsample_hover_data = hover_data.iloc[subsample_indices]
#subsample_labels = morph_labels[subsample_indices]
# Ensure subsample_labels has the same length as subsample_hover_data
#subsample_labels = subsample_labels[:len(subsample_hover_data)]
# Create the interactive plot with subsampled data
#interactive_plot = umap.plot.interactive(mapper, labels=subsample_labels, hover_data=subsample_hover_data, point_size=2)
interactive_plot = umap.plot.interactive(mapper,subset_points=ss, hover_data=hover_data, point_size=2)
umap.plot.show(interactive_plot)
#umap.plot.output_file("interactive_plot.html")
#umap.plot.show(interactive_plot, browser=False)


In [ ]:
morph_fitness

In [ ]:
%%time
mapper_euclidean = umap.UMAP(metric='euclidean', low_memory=True).fit(morph_sparse)

In [ ]:
umap.plot.points(mapper_euclidean, labels=morph_labels, theme='inferno') #values=np.arange(morph_sparse.shape[0]), )

In [ ]:
%%time
mapper_canberra = umap.UMAP(metric='canberra', low_memory=True).fit(morph_sparse)

In [ ]:
umap.plot.points(mapper_canberra, labels=morph_labels, theme='inferno') #values=np.arange(morph_sparse.shape[0]), )

In [ ]:
%%time
mapper_manhattan = umap.UMAP(metric='manhattan', low_memory=True).fit(morph_sparse)

In [ ]:
umap.plot.points(mapper_manhattan, labels=morph_labels, theme='inferno') #values=np.arange(morph_sparse.shape[0]), )

In [ ]:
%%time
mapper_manhattan = umap.UMAP(metric='manhattan', low_memory=True, n_neighbors=5).fit(morph_sparse)

In [ ]:
umap.plot.points(mapper_manhattan, labels=morph_labels, theme='inferno') #values=np.arange(morph_sparse.shape[0]), )

In [ ]:
%%time
mapper_manhattan3 = umap.UMAP(metric='manhattan', low_memory=True, n_neighbors=20, min_dist=0).fit(morph_sparse)

In [ ]:
umap.plot.points(mapper_manhattan3, labels=morph_labels, theme='inferno') #values=np.arange(morph_sparse.shape[0]), )

In [ ]:
%%time
mapper_manhattan4 = umap.UMAP(metric='manhattan', low_memory=True, n_neighbors=30, min_dist=0).fit(morph_sparse)

In [ ]:
umap.plot.points(mapper_manhattan4, labels=morph_labels, theme='inferno') #values=np.arange(morph_sparse.shape[0]), )

In [ ]:
%%time
mapper_manhattan5 = umap.UMAP(metric='manhattan', low_memory=True, n_neighbors=40, min_dist=0).fit(morph_sparse)

In [ ]:
umap.plot.points(mapper_manhattan5, labels=morph_labels, theme='inferno') #values=np.arange(morph_sparse.shape[0]), )

In [ ]:
%%time
mapper_manhattan6 = umap.UMAP(metric='manhattan', low_memory=True, n_neighbors=60, min_dist=0).fit(morph_sparse)

In [ ]:
umap.plot.points(mapper_manhattan6, labels=morph_labels, theme='inferno') #values=np.arange(morph_sparse.shape[0]), )

In [ ]:
ss = np.greater(morph_fitness, 0.9)
umap.plot.points(mapper_manhattan6, labels=morph_labels, subset_points=ss, theme='inferno',alpha=0.9) #values=np.arange(morph_sparse.shape[0]), )
the_plot.set_xlim(-0.5,799.5)
the_plot.set_ylim(799.5,-0.5)